In [ ]:
import json
import torch
import matplotlib.pyplot as plt

from bnbp5.bnn_intralayer import *
from bnbp5.trainnospikemne_intralayer_timeind_sliding_128 import Trainer
from bnbp5.mnist_spiketrain_sliding import *
import numpy as np
import matplotlib.pyplot as plt
from os.path import exists

from zanj import ZANJ

from torchvision import datasets, transforms
from torch import nn

torch.set_default_dtype(torch.float32)

In [ ]:
_HH_PARAMS: dict[str, float] = {
    "gna": 40.0,
    "gk": 35.0,
    "gl": 0.4, #0.3, 0.4

    "Ena": 55.0,
    "Ek": -77.0,
    "El": -60.0, #-65.0,

    "gm": 0.075,
    "ghca": 0.12, #12.0, #changed 0.12,
    "vthresh": -56.2, 
    "tau_max": 0.608,
    "Eca":  55.0, #120120.0,
    
    "gs": 0.04,
    "Vs": 0.0,
    "Iapp": 0.7,
    "lat_inhibition": False,
    "beta_n_modified": False,

    "Vt": -3.0,
    "Kp": 8.0,
    "a_d": 1.0,
    "a_r": 0.1,   
}   

# Adjust for DNN/BiLSTM/SNN
CFG1: BNNConfig = BNNConfig(lr = 0.0001, neuron_model = model_HH_RS, test_batch_sz = 25, train_batch_sz = 20, neuron_params = _HH_PARAMS, model_dims = [128,100,2], dt = 1000/220/45, plot_interm=True)
    
CFG2: DatasetConfig = DatasetConfig(
            sim_t = 2000,n_samples_train=300, n_samples_val = 50, n_samples_test=9000)
    

torch.manual_seed(15)
    
trainer = Trainer(CFG1, CFG2, False,subjects=["UM_7"], num_classes=2)    

In [ ]:
'''
state = torch.load(  'BNN_Anesthesia_Sub007.pth',map_location=torch.device('cuda'))

model_state_dict = state['model_state_dict']
optimizer_state_dict = state['optimizer_state_dict']

trainer.model.load_state_dict(model_state_dict)
#trainer.optimizer.state_dict(optimizer_state_dict)

#for param_group in trainer.optimizer.param_groups:
#    param_group['lr'] = param_group['lr']*0.1
'''

In [ ]:

accuracies_epochs= []#state['accuracies_epochs']
loss_epochs = []# state['loss']
#epch = state['epoch']

# With accuracies and loss_record
trainer.optimizer.param_groups[0]['lr'] = 0.00001

for epch in range(20):
        print("epoch", epch)
        accuracies, loss_record = trainer.train(epch, 14)
        
        accuracies_epochs.append(accuracies)
        loss_epochs.append(loss_record)
                    
        torch.save({
        'epoch': epch,
        'model_state_dict': trainer.model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'accuracies_epochs': accuracies_epochs,
        'loss': loss_epochs,
        'CFG1':CFG1,
        'CFG2':CFG2,
        }, 'BNN_Anesthesia_Sub007_2states.pth')

    
print("DONE")